# Results for model: microsoft/phi-4-multimodal-instruct

In [ ]:
import polars as pol
from polars.cleaning import quantile
from polars.ml.features import quantile
from polars.ml.stratified_mean_groupby import stratified_mean_groupby
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split

# Load the Data
df = pol.DataFrame.read_parquet("./jane_street_data/train.parquet")

# Feature Engineering: Calculate global TOP_QUANTILE (top 15%) of 'feature_00' within each the rolling batches of 'date_id'
df = df.lazy()
df = df.with_stage('df')
df = df.groupby('date_id').agg([
    quantile(df['feature_00'], 0.85).alias("feature_00_quantile_0.85"),
]).coalesce(1)

# Split data into training and test sets
train_df, test_df = df.sample(seed=42).collect(), df.filter(df['date_id'] != 'train').collect()

# Further split into X and y
X_train = train_df[['feature_00_quantile_0.85']]
y_train = train_df['responder_6']
X_test = test_df[['feature_00_quantile_0.85']]
y_test = test_df['responder_6']

# Convert to Polars DataFrame type again to maintain compatibility
X_train = X_train.to_polars()
y_train = y_train.to_polars()

# Train the initial model
model = XGBRegressor()
model.fit(X_train, y_train)

# Evaluate the model
y_train_pred = model.predict(X_train)
train_rmse = mean_squared_error(y_train.collect_list(), y_train_pred.collect_list(), squared=False)
print(f"Train RMSE: {train_rmse}")

y_test_pred = model.predict(X_test)
test_rmse = mean_squared_error(y_test.collect_list(), y_test_pred.collect_list(), squared=False)
print(f"Test RMSE: {test_rmse}")